# Unsupervised Learning Lab

This notebook covers clustering, dimensionality reduction, and visualization techniques.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_blobs, make_moons, make_circles, load_digits
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
%matplotlib inline

## 1. Generate Synthetic Clustering Data

In [ ]:
# Generate different types of clustering data
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, X, y, title in zip(axes, [X_blobs, X_moons, X_circles], 
                            [y_blobs, y_moons, y_circles],
                            ['Blobs', 'Moons', 'Circles']):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=20)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 2. K-Means from Scratch

In [ ]:
class KMeansFromScratch:
    def __init__(self, n_clusters=3, max_iters=100, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
        self.history = []  # store centroids at each step
    
    def fit(self, X):
        n_samples = X.shape[0]
        # Initialize centroids randomly from data points
        idx = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[idx].copy()
        
        for i in range(self.max_iters):
            # Assign clusters
            distances = self._compute_distances(X)
            self.labels = np.argmin(distances, axis=1)
            self.history.append(self.centroids.copy())
            
            # Update centroids
            new_centroids = np.array([X[self.labels == k].mean(axis=0) 
                                      for k in range(self.n_clusters)])
            
            # Check convergence
            if np.all(np.abs(new_centroids - self.centroids) < self.tol):
                break
            self.centroids = new_centroids
        
        return self
    
    def _compute_distances(self, X):
        distances = np.zeros((X.shape[0], self.n_clusters))
        for k in range(self.n_clusters):
            distances[:, k] = np.linalg.norm(X - self.centroids[k], axis=1)
        return distances
    
    def predict(self, X):
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)

In [ ]:
# Run K-Means from scratch and visualize iterations
kmeans_scratch = KMeansFromScratch(n_clusters=4)
kmeans_scratch.fit(X_blobs)

# Visualize first 4 iterations
fig, axes = plt.subplots(1, min(4, len(kmeans_scratch.history)), figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans_scratch.labels, 
              cmap='viridis', alpha=0.5, s=20)
    centroids = kmeans_scratch.history[i]
    ax.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', s=200, 
              edgecolors='black', linewidth=2)
    ax.set_title(f'Iteration {i+1}')
plt.tight_layout()
plt.show()

## 3. Elbow Method and Silhouette Analysis

In [ ]:
# Elbow Method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].axvline(x=4, color='r', linestyle='--', label='Optimal K=4')
axes[0].legend()

# Silhouette scores
sil_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    sil_scores.append(silhouette_score(X_blobs, labels))

axes[1].plot(K_range, sil_scores, 'go-')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].axvline(x=4, color='r', linestyle='--', label='Optimal K=4')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Detailed silhouette plot for K=4
from sklearn.cluster import KMeans
km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X_blobs)
sample_sil = silhouette_samples(X_blobs, labels)

fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10
for i in range(4):
    cluster_sil = sample_sil[labels == i]
    cluster_sil.sort()
    size = cluster_sil.shape[0]
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size, str(i))
    y_lower = y_upper + 10

ax.axvline(x=silhouette_score(X_blobs, labels), color='red', linestyle='--')
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Cluster')
ax.set_title('Silhouette Plot for K=4')
plt.show()

## 4. DBSCAN - Where K-Means Fails

In [ ]:
# K-Means vs DBSCAN on non-convex data
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
datasets = [(X_blobs, 'Blobs'), (X_moons, 'Moons'), (X_circles, 'Circles')]

for i, (X, name) in enumerate(datasets):
    # K-Means
    km_labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X)
    axes[0, i].scatter(X[:, 0], X[:, 1], c=km_labels, cmap='viridis', s=20)
    axes[0, i].set_title(f'K-Means on {name}')
    
    # DBSCAN
    db_labels = DBSCAN(eps=0.3 if name != 'Blobs' else 0.5, min_samples=5).fit_predict(X)
    axes[1, i].scatter(X[:, 0], X[:, 1], c=db_labels, cmap='viridis', s=20)
    axes[1, i].set_title(f'DBSCAN on {name}')

plt.tight_layout()
plt.show()
print('Note: K-Means fails on moons and circles because it assumes convex clusters')

## 5. PCA Step-by-Step

In [ ]:
# PCA from scratch
np.random.seed(42)
# Generate correlated 2D data
mean = [0, 0]
cov = [[3, 2], [2, 2]]
X_pca = np.random.multivariate_normal(mean, cov, 200)

# Step 1: Center the data
X_centered = X_pca - X_pca.mean(axis=0)
print('Step 1: Data centered (mean subtracted)')
print(f'Mean after centering: {X_centered.mean(axis=0)}')

# Step 2: Compute covariance matrix
cov_matrix = np.cov(X_centered.T)
print(f'\nStep 2: Covariance matrix:\n{cov_matrix}')

# Step 3: Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
print(f'\nStep 3: Eigenvalues: {eigenvalues}')
print(f'Eigenvectors:\n{eigenvectors}')

# Step 4: Sort by eigenvalue
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]
print(f'\nStep 4: Sorted eigenvalues: {eigenvalues}')
print(f'Explained variance ratio: {eigenvalues / eigenvalues.sum()}')

In [ ]:
# Visualize PCA components
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original data with eigenvectors
axes[0].scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5, s=20)
for i in range(2):
    axes[0].arrow(0, 0, eigenvectors[0, i]*eigenvalues[i]*0.5, 
                  eigenvectors[1, i]*eigenvalues[i]*0.5,
                  head_width=0.15, color=['red', 'blue'][i], linewidth=2)
axes[0].set_title('Data with Principal Components')
axes[0].set_aspect('equal')
axes[0].legend(['Data', 'PC1', 'PC2'])

# Project onto first PC
X_projected = X_centered @ eigenvectors[:, :1]
X_reconstructed = X_projected @ eigenvectors[:, :1].T
axes[1].scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.3, s=20, label='Original')
axes[1].scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], alpha=0.5, s=20, label='Projected to 1D')
axes[1].set_title('Projection onto First Principal Component')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. PCA on Digits Dataset

In [ ]:
# Load digits dataset
digits = load_digits()
X_digits = digits.data
y_digits = digits.target
print(f'Digits shape: {X_digits.shape} (64 features = 8x8 pixels)')

# Apply PCA
pca = PCA()
X_pca_digits = pca.fit_transform(X_digits)

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(np.cumsum(pca.explained_variance_ratio_), 'b-')
axes[0].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Cumulative Explained Variance')
axes[0].set_title('PCA: Explained Variance vs Components')
axes[0].legend()

# 2D projection
scatter = axes[1].scatter(X_pca_digits[:, 0], X_pca_digits[:, 1], 
                          c=y_digits, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, ax=axes[1])
axes[1].set_title('Digits: PCA 2D Projection')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.tight_layout()
plt.show()

n_95 = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1
print(f'Components needed for 95% variance: {n_95}')

## 7. t-SNE vs PCA Comparison

In [ ]:
# t-SNE on digits
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne_digits = tsne.fit_transform(X_digits)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA
scatter1 = axes[0].scatter(X_pca_digits[:, 0], X_pca_digits[:, 1], 
                           c=y_digits, cmap='tab10', s=10, alpha=0.7)
axes[0].set_title('PCA (Linear)')
axes[0].set_xlabel('Component 1')
axes[0].set_ylabel('Component 2')
plt.colorbar(scatter1, ax=axes[0])

# t-SNE
scatter2 = axes[1].scatter(X_tsne_digits[:, 0], X_tsne_digits[:, 1], 
                           c=y_digits, cmap='tab10', s=10, alpha=0.7)
axes[1].set_title('t-SNE (Non-linear)')
axes[1].set_xlabel('Dimension 1')
axes[1].set_ylabel('Dimension 2')
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.show()
print('t-SNE preserves local structure much better than PCA for visualization')

In [ ]:
# Effect of perplexity on t-SNE
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
perplexities = [5, 15, 30, 50]
for ax, perp in zip(axes, perplexities):
    tsne_p = TSNE(n_components=2, random_state=42, perplexity=perp)
    X_t = tsne_p.fit_transform(X_digits)
    ax.scatter(X_t[:, 0], X_t[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f'Perplexity = {perp}')
plt.suptitle('t-SNE: Effect of Perplexity', y=1.02)
plt.tight_layout()
plt.show()

## Summary

| Method | Type | Strengths | Weaknesses |
|--------|------|-----------|------------|
| K-Means | Clustering | Fast, scalable | Assumes convex clusters, needs K |
| DBSCAN | Clustering | Finds arbitrary shapes, no K needed | Sensitive to eps |
| PCA | Dim. Reduction | Linear, fast, interpretable | Misses non-linear structure |
| t-SNE | Dim. Reduction | Great visualization | Slow, non-parametric |